# Text-Only Baseline — transformer classifier

Fine-tunes a DistilBERT/BERT-family transformer for binary sequence classification.

**Purpose:** establish what text alone achieves, so the multimodal model has a meaningful target to
beat. Without this, "93.94% accuracy" says nothing about whether vision contributed.

**Setup:** `AutoModelForSequenceClassification`, attention dropout 0.2, hidden dropout 0.2,
HuggingFace `Trainer`. Trained on the public FRENK hate-speech benchmark (LGBT subset).

**Result:** ~86.87% accuracy.

**Known limitation — important:** this baseline trains on FRENK, *not* on meme OCR captions. The
text-vs-multimodal comparison therefore varies both modality and training corpus, so the ~7-point
multimodal gain is indicative rather than a controlled ablation.


In [ ]:
pip install transformers datasets torch scikit-learn


In [ ]:
from datasets import load_dataset

# Load the FRENK-hate-en dataset
dataset = load_dataset("classla/FRENK-hate-en")

# Display dataset structure
print("Dataset Structure:", dataset)

# Check the columns in the dataset and some examples
print("Train Dataset Columns:", dataset['train'].column_names)
print("Validation Dataset Columns:", dataset['validation'].column_names)
print("Test Dataset Columns:", dataset['test'].column_names)

# Show a sample from the dataset
print("Train Dataset Sample:", dataset['train'][0])

# Filter dataset to include only 'lgbt' topic
lgbt_dataset = dataset.filter(lambda x: x['topic'] == 'lgbt')

# Display the number of samples in the filtered dataset
print("Number of samples in 'lgbt' topic dataset:")
print(f"Train: {len(lgbt_dataset['train'])}")
print(f"Validation: {len(lgbt_dataset['validation'])}")
print(f"Test: {len(lgbt_dataset['test'])}")

# Check the class distribution in the 'label' column
print("Label Distribution in 'train' dataset:")
print(lgbt_dataset['train']['label'])

# You can also check how many unique labels exist
unique_labels = set(lgbt_dataset['train']['label'])
print(f"Unique labels: {unique_labels}")

# Print a few examples of data
print("\nFew examples of 'lgbt' data in train set:")
for i in range(5):
    print(f"Example {i + 1}: {lgbt_dataset['train'][i]}")


In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer

# Load the dataset
dataset = load_dataset('classla/FRENK-hate-en')

# Load a pre-trained tokenizer (you can change this to any relevant model)
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

# Function to preprocess the data
def preprocess_function(examples):
    # Tokenize the text using the tokenizer
    return tokenizer(examples['text'], padding='max_length', truncation=True, max_length=512)

# Apply the preprocessing to each split
encoded_dataset = dataset.map(preprocess_function, batched=True)

# Check a sample from the processed dataset
print(encoded_dataset["train"][0])


In [ ]:
# from transformers import Trainer, TrainingArguments, AutoModelForSequenceClassification, EarlyStoppingCallback
# import torch

# # Load the model for sequence classification (DistilBERT)
# model = AutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=2)

# # Define the training arguments
# training_args = TrainingArguments(
#     output_dir='./results',              # output directory for the model and logs
#     eval_strategy="epoch",               # evaluation strategy (run evaluation at the end of each epoch)
#     save_strategy="epoch",               # save strategy (save the model at the end of each epoch)
#     learning_rate=2e-5,                  # learning rate for the optimizer
#     per_device_train_batch_size=8,       # batch size for training
#     per_device_eval_batch_size=8,        # batch size for evaluation
#     num_train_epochs=3,                  # number of training epochs
#     weight_decay=0.01,                   # weight decay for optimization
#     logging_dir='./logs',                # directory to store logs
#     logging_steps=10,                    # log every 10 steps
#     load_best_model_at_end=True,         # Load the best model when finished
# )

# # Initialize EarlyStoppingCallback
# early_stopping_callback = EarlyStoppingCallback(early_stopping_patience=3)

# # Define the Trainer
# trainer = Trainer(
#     model=model,
#     args=training_args,
#     train_dataset=encoded_dataset['train'],
#     eval_dataset=encoded_dataset['validation'],
#     tokenizer=tokenizer,
#     callbacks=[early_stopping_callback],  # Include the EarlyStoppingCallback
# )

# # Train the model
# trainer.train()

# # Save the trained model and tokenizer after training
# model_save_path = './saved_model'
# model.save_pretrained(model_save_path)
# tokenizer.save_pretrained(model_save_path)

# print(f"Model saved to {model_save_path}")


In [ ]:
!pip install optuna


In [ ]:
import optuna
from transformers import Trainer, TrainingArguments, AutoModelForSequenceClassification
import torch

# Step 1: Define the objective function for Optuna
def objective(trial):
    # Hyperparameter search space
    learning_rate = trial.suggest_categorical('learning_rate', [1e-5, 2e-5, 3e-5, 1e-4, 2e-4])
    batch_size = trial.suggest_categorical('batch_size', [8, 16])
    epochs = trial.suggest_categorical('epochs', [3])
    dropout_rate = trial.suggest_categorical('dropout_rate', [0.1, 0.2, 0.3])
    weight_decay = trial.suggest_categorical('weight_decay', [0.01, 0.1])

    # Load the model with the dropout rate from the hyperparameter space
    model = AutoModelForSequenceClassification.from_pretrained('distilbert-base-uncased', num_labels=2)

    # Apply dropout to the model
    model.config.attention_probs_dropout_prob = dropout_rate
    model.config.hidden_dropout_prob = dropout_rate

    # Define training arguments
    training_args = TrainingArguments(
        output_dir='./results',              # output directory for the model and logs
        learning_rate=learning_rate,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size,
        num_train_epochs=epochs,
        weight_decay=weight_decay,
        logging_dir='./logs',
        evaluation_strategy="epoch",         # evaluate at the end of each epoch
        save_strategy="epoch",               # save after each epoch
        logging_steps=10,
    )

    # Create Trainer with the current hyperparameters
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=encoded_dataset['train'],  # Replace with your actual train dataset
        eval_dataset=encoded_dataset['validation'],  # Replace with your actual validation dataset
    )

    # Train the model
    trainer.train()

    # Evaluate the model and return the evaluation loss
    eval_results = trainer.evaluate()

    # Return the evaluation loss (or you could return accuracy, F1-score, etc.)
    return eval_results['eval_loss']  # You can return other metrics if preferred

# Step 2: Create an Optuna study and optimize the objective function
study = optuna.create_study(direction='minimize')  # We want to minimize the loss
study.optimize(objective, n_trials=50)  # Perform 50 trials, you can adjust as needed

# Step 3: Print the best hyperparameters found by Optuna
print(f"Best hyperparameters: {study.best_params}")


[I 2024-11-29 21:23:33,111] A new study created in memory with name: no-name-ad730aca-4012-496d-ac25-64ff8e7c3f0e
Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/usr/local/lib/python3.10/dist-packages/transformers/training_args.py:1568: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
 [5255/5255 33:50, Epoch 5/5]
Epoch	Training Loss	Validation Loss
1	0.627000	0.661510
2	0.687800	0.650304
3	0.634700	0.662913
4	0.561800	0.596992
5	0.662800	0.599255
 [117/117 00:12]
[I 2024-11-29 21:57:37,994] Trial 0 finished with value: 0.5992550849914551 and parameters: {'learning_rate': 0.0002, 'batch_size': 8, 'epochs': 5, 'dropout_rate': 0.3, 'weight_decay': 0.01}. Best is trial 0 with value: 0.5992550849914551.
Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/usr/local/lib/python3.10/dist-packages/transformers/training_args.py:1568: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
 [10510/10510 1:08:21, Epoch 10/10]
Epoch	Training Loss	Validation Loss
1	0.379300	0.468694
2	0.371000	0.559220
3	0.032600	1.026544
4	0.170900	1.271367
5	0.007100	1.620270
6	0.141200	1.807019
7	0.000100	1.719407
8	0.000100	1.889379
9	0.000000	2.008767
10	0.000000	1.994293
 [117/117 00:13]
[I 2024-11-29 23:06:13,212] Trial 1 finished with value: 1.994292974472046 and parameters: {'learning_rate': 3e-05, 'batch_size': 8, 'epochs': 10, 'dropout_rate': 0.3, 'weight_decay': 0.1}. Best is trial 0 with value: 0.5992550849914551.
Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/usr/local/lib/python3.10/dist-packages/transformers/training_args.py:1568: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
 [5255/5255 33:34, Epoch 5/5]
Epoch	Training Loss	Validation Loss
1	0.396000	0.466286
2	0.372000	0.572296
3	0.093000	0.875004
4	0.171300	1.088232

 [ 82/117 00:08 < 00:03, 9.03 it/s]

In [ ]:
from transformers import Trainer, TrainingArguments, AutoModelForSequenceClassification, EarlyStoppingCallback
import torch

# Load the model for sequence classification (DistilBERT)
model = AutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=2)

# Modify dropout settings (can be adjusted as per requirement)
model.config.attention_dropout = 0.3  # Increase attention dropout
model.config.hidden_dropout = 0.3    # Increase hidden layer dropout

# Define the training arguments
training_args = TrainingArguments(
    output_dir='./results',              # output directory for the model and logs
    eval_strategy="epoch",               # evaluation strategy (run evaluation at the end of each epoch)
    save_strategy="epoch",               # save strategy (save the model at the end of each epoch)
    learning_rate=0.0002,                  # learning rate for the optimizer
    per_device_train_batch_size=8,       # batch size for training
    per_device_eval_batch_size=8,        # batch size for evaluation
    num_train_epochs=5,                  # number of training epochs
    weight_decay=0.01,                   # weight decay for optimization
    logging_dir='./logs',                # directory to store logs
    logging_steps=10,                    # log every 10 steps
    load_best_model_at_end=True,         # Load the best model when finished
    max_grad_norm=1.0,                   # Apply gradient clipping with value 1.0 (adjust as needed)
)

# Initialize EarlyStoppingCallback
early_stopping_callback = EarlyStoppingCallback(early_stopping_patience=3)

# Define the Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=encoded_dataset['train'],
    eval_dataset=encoded_dataset['validation'],
    tokenizer=tokenizer,
    callbacks=[early_stopping_callback],  # Include the EarlyStoppingCallback
)

# Train the model
trainer.train()

# Save the trained model and tokenizer after training
model_save_path = './saved_model'
model.save_pretrained(model_save_path)
tokenizer.save_pretrained(model_save_path)

print(f"Model saved to {model_save_path}")


In [ ]:
from transformers import Trainer, TrainingArguments, AutoModelForSequenceClassification, EarlyStoppingCallback
import torch

# Load the model for sequence classification (DistilBERT)
model = AutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=2)

# Modify dropout settings (can be adjusted as per requirement)
model.config.attention_dropout = 0.3  # Increase attention dropout
model.config.hidden_dropout = 0.3    # Increase hidden layer dropout

# Define the training arguments
training_args = TrainingArguments(
    output_dir='./results',              # output directory for the model and logs
    eval_strategy="epoch",               # evaluation strategy (run evaluation at the end of each epoch)
    save_strategy="epoch",               # save strategy (save the model at the end of each epoch)
    learning_rate=0.0002,                  # learning rate for the optimizer
    per_device_train_batch_size=8,       # batch size for training
    per_device_eval_batch_size=8,        # batch size for evaluation
    num_train_epochs=5,                  # number of training epochs
    weight_decay=0.01,                   # weight decay for optimization
    logging_dir='./logs',                # directory to store logs
    logging_steps=10,                    # log every 10 steps
    load_best_model_at_end=True,         # Load the best model when finished
    max_grad_norm=1.0,                   # Apply gradient clipping with value 1.0 (adjust as needed)
)

# Initialize EarlyStoppingCallback
early_stopping_callback = EarlyStoppingCallback(early_stopping_patience=3)

# Define the Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=encoded_dataset['train'],
    eval_dataset=encoded_dataset['validation'],
    tokenizer=tokenizer,
    callbacks=[early_stopping_callback],  # Include the EarlyStoppingCallback
)

# Train the model
trainer.train()

# Save the trained model and tokenizer after training
model_save_path = './saved_model'
model.save_pretrained(model_save_path)
tokenizer.save_pretrained(model_save_path)

print(f"Model saved to {model_save_path}")

In [ ]:
import torch, numpy as np
from sklearn.metrics import confusion_matrix, classification_report

# Define the device (GPU or CPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Move the model to the correct device
model.to(device)

# Evaluation function
def evaluate_model(trainer):
    # Set the model to evaluation mode
    model.eval()

    all_preds = []
    all_labels = []

    # Iterate through the evaluation dataset
    with torch.no_grad():
        for batch in trainer.get_eval_dataloader():
            # Move the batch to the same device as the model
            inputs = {key: val.to(device) for key, val in batch.items()}

            # Get the model outputs
            outputs = model(**inputs)
            logits = outputs.logits

            # Get the predictions (index of the maximum logit)
            preds = torch.argmax(logits, dim=-1)

            # Store predictions and true labels
            all_preds.extend(preds.cpu().numpy())  # Move predictions to CPU
            all_labels.extend(inputs['labels'].cpu().numpy())  # Move labels to CPU

    # Convert to numpy arrays for sklearn functions
    all_preds = np.array(all_preds)
    all_labels = np.array(all_labels)

    # Generate confusion matrix and classification report
    print("Confusion Matrix:")
    print(confusion_matrix(all_labels, all_preds))

    print("\nClassification Report:")
    print(classification_report(all_labels, all_preds))

# Evaluate the model
evaluate_model(trainer)


# Manual error analysis (look at misclassified examples)
def manual_error_analysis(model, eval_dataloader):
    incorrect_preds = []
    model.eval()
    with torch.no_grad():
        for batch in eval_dataloader:
            inputs = {key: val.to(device) for key, val in batch.items()}
            outputs = model(**inputs)
            logits = outputs.logits
            preds = torch.argmax(logits, dim=-1).cpu().numpy()
            true_labels = inputs['labels'].cpu().numpy()

            # Collect misclassified examples
            for i, (true_label, pred_label) in enumerate(zip(true_labels, preds)):
                if true_label != pred_label:
                    incorrect_preds.append({
                        "text": inputs['input_ids'][i],  # Store text here, ensure you decode it if needed
                        "true_label": true_label,
                        "pred_label": pred_label
                    })

    # Review misclassified examples
    for item in incorrect_preds[:5]:  # Limit to first 5 misclassified examples
        print(f"Text: {item['text']}")
        print(f"True Label: {item['true_label']} | Predicted Label: {item['pred_label']}")

# Perform manual error analysis on the evaluation dataset
manual_error_analysis(model, trainer.get_eval_dataloader())


In [ ]:
# Save the trained model and tokenizer
model_save_path = './saved_model'

# Save the model
model.save_pretrained(model_save_path)

# Save the tokenizer (in case you want to use the same tokenizer later)
tokenizer.save_pretrained(model_save_path)

print(f"Model saved to {model_save_path}")
